# Lab 08 — On-Policy Gameplay SFT

**Goal:** fix the Lab 07 transfer failure by training on the exact gameplay states the model encounters under the frozen Lab 04 interface.

Lab 07 learned auxiliary Wordle skills but actual gameplay stayed near baseline. The likely problem is distribution mismatch: Lab 07 trained on expert trajectories and task wrappers, while deployment uses the Lab 04 gameplay prompt and states produced by the learner itself.

This lab uses a DAgger-style loop:

1. load the Lab 07 checkpoint;
2. let it play real games with the exact Lab 04 prompt;
3. record each state it reaches;
4. relabel that state with the symbolic expert;
5. fine-tune on those exact deployment prompt → expert action pairs;
6. rerun the frozen benchmark.

Do one round first. Measure it before adding complexity.

In [1]:
from __future__ import annotations

from dataclasses import dataclass, asdict
from pathlib import Path
from collections import Counter
import hashlib, json, math, random, re, time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch.optim import AdamW
from torch.utils.data import DataLoader
from transformers import AutoModelForCausalLM, AutoTokenizer

from tiny_wordle.game import Turn, score_string
from tiny_wordle.expert import EntropyExpert

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)
print("PyTorch:", torch.__version__)

device: mps
PyTorch: 2.13.0


## 8.1 Load the Lab 07 checkpoint

In [2]:
LAB07_CHECKPOINT = Path("../checkpoints/qwen3-0.6b-wordle-full-sft")

tokenizer = AutoTokenizer.from_pretrained(LAB07_CHECKPOINT)
model = AutoModelForCausalLM.from_pretrained(
    LAB07_CHECKPOINT,
    dtype=torch.float32,
).to(device)
model.eval()

print(type(model).__name__)
print("parameters:", f"{sum(p.numel() for p in model.parameters()):,}")
print("device:", next(model.parameters()).device)

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Qwen3ForCausalLM
parameters: 596,049,920
device: mps:0


## 8.2 Load the symbolic expert

In [3]:
DATA_DIR = Path("../data")

ANSWERS = [
    x.strip().upper()
    for x in (DATA_DIR / "wordle-answers-original.txt").read_text().splitlines()
    if x.strip()
]
PATTERNS = np.load(DATA_DIR / "wordle-patterns-original-2315.npy")

expert = EntropyExpert(ANSWERS, PATTERNS)
WORD_TO_INDEX = expert.word_to_index
ALL_INDICES = expert.all_indices

print("answers:", len(ANSWERS))
print("pattern matrix:", PATTERNS.shape)

answers: 2315
pattern matrix: (2315, 2315)


## 8.3 Freeze the exact Lab 04 gameplay prompt

Do not tune this prompt in Lab 08. The model must adapt to deployment, not the other way around.

In [5]:
SYSTEM_RULES = '''Play Wordle.

Return exactly one uppercase five-letter English word.
Do not explain.
Do not use punctuation.

Use all previous guesses and feedback when choosing the next guess.
Never repeat a previous guess.

Example valid response:
CRANE

Feedback meanings:
G = correct letter and position
Y = letter is present but wrong position
B = that letter occurrence is not matched
'''

def format_history_for_model(history: list[Turn]) -> str:
    if not history:
        return "No guesses have been made yet."
    return "\n".join(
        f"{' '.join(t.guess)} -> {' '.join(t.feedback)}"
        for t in history
    )

def build_prompt(history: list[Turn]) -> str:
    return SYSTEM_RULES + "\nGame history:\n" + format_history_for_model(history)

WORD_RE = re.compile(r"^[A-Za-z]{5}$")

def parse_guess(raw_text: str) -> str | None:
    text = raw_text.strip()
    return text.upper() if WORD_RE.fullmatch(text) else None

## 8.4 Deterministic model generation

In [7]:
def render_prompt(prompt: str) -> str:
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

def generate_raw_guess(history: list[Turn], max_new_tokens: int = 16) -> str:
    text = render_prompt(build_prompt(history))
    batch = tokenizer(text, return_tensors="pt").to(device)
    model.eval()
    with torch.no_grad():
        output = model.generate(
            **batch,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )
    new = output[0, batch["input_ids"].shape[1]:]
    return tokenizer.decode(new, skip_special_tokens=True).strip()

## 8.5 Recover the exact candidate set from an arbitrary learner history

The expert must relabel states produced by the learner, not assume its own trajectory.

In [8]:
MARK_DIGIT = {"B": 0, "Y": 1, "G": 2}

def encode_feedback(feedback: str) -> int:
    value = 0
    for mark in feedback:
        value = value * 3 + MARK_DIGIT[mark]
    return value

def candidate_indices_from_history(history: list[Turn]) -> np.ndarray:
    candidates = ALL_INDICES.copy()

    for turn in history:
        if turn.guess in WORD_TO_INDEX:
            guess_idx = WORD_TO_INDEX[turn.guess]
            pattern_id = encode_feedback(turn.feedback)
            candidates = candidates[
                PATTERNS[guess_idx, candidates] == pattern_id
            ]
        else:
            surviving = [
                int(idx)
                for idx in candidates
                if score_string(ANSWERS[int(idx)], turn.guess) == turn.feedback
            ]
            candidates = np.array(surviving, dtype=np.int32)

        if len(candidates) == 0:
            break

    return candidates

def expert_action_for_history(history: list[Turn]) -> tuple[str | None, int]:
    candidates = candidate_indices_from_history(history)
    if len(candidates) == 0:
        return None, 0

    seen = {t.guess for t in history}
    available = np.array(
        [int(i) for i in candidates if ANSWERS[int(i)] not in seen],
        dtype=np.int32,
    )
    if len(available) == 0:
        return None, len(candidates)

    guess_idx = expert.choose(available)
    return ANSWERS[guess_idx], len(candidates)

## 8.6 Smoke-test expert relabeling on the known bad state

In [9]:
smoke_history = [Turn("CRANE", "BBGGB")]
expert_guess, candidate_count = expert_action_for_history(smoke_history)

print("candidate count:", candidate_count)
print("expert action:", expert_guess)
print("repeat:", expert_guess == "CRANE")

candidate count: 16
expert action: STANK
repeat: False


## 8.7 Reuse the Lab 06 answer split

In [10]:
DEV_FIXED = {"PLANT"}

TEST_FIXED = {
    "SHORE", "MIGHT", "BRICK", "GHOST", "KNIFE",
    "DOUBT", "FLING", "ROUND", "CHAMP", "WASTE",
    "BLIND", "POINT", "SLATE", "CRANE", "APPLE",
    "SHEEP", "BANAL", "ALLEY", "AUDIO",
}

def stable_bucket(word: str) -> int:
    digest = hashlib.sha256(word.encode("utf-8")).digest()
    return int.from_bytes(digest[:8], "big") % 1000

def answer_split(answer: str) -> str:
    if answer in TEST_FIXED:
        return "test"
    if answer in DEV_FIXED:
        return "dev"
    return "dev" if stable_bucket(answer) < 100 else "train"

print(Counter(answer_split(w) for w in ANSWERS))

Counter({'train': 2065, 'dev': 231, 'test': 19})


## 8.8 Collect exact learner states

In [11]:
@dataclass
class OnPolicyState:
    answer: str
    split: str
    turn: int
    prompt: str
    history: list[dict]
    learner_raw: str
    learner_guess: str | None
    expert_guess: str | None
    candidate_count: int
    learner_repeated: bool
    learner_feedback: str | None

def collect_game_states(answer: str, max_turns: int = 6) -> list[OnPolicyState]:
    history: list[Turn] = []
    seen = set()
    records = []

    for turn_number in range(1, max_turns + 1):
        prompt = build_prompt(history)
        expert_guess, candidate_count = expert_action_for_history(history)

        raw = generate_raw_guess(history)
        learner_guess = parse_guess(raw)
        repeated = learner_guess in seen if learner_guess is not None else False

        record = OnPolicyState(
            answer=answer,
            split=answer_split(answer),
            turn=turn_number,
            prompt=prompt,
            history=[{"guess": t.guess, "feedback": t.feedback} for t in history],
            learner_raw=raw,
            learner_guess=learner_guess,
            expert_guess=expert_guess,
            candidate_count=candidate_count,
            learner_repeated=repeated,
            learner_feedback=None,
        )
        records.append(record)

        if learner_guess is None:
            continue

        seen.add(learner_guess)
        feedback = score_string(answer, learner_guess)
        records[-1].learner_feedback = feedback
        history.append(Turn(learner_guess, feedback))

        if feedback == "GGGGG":
            break

    return records

## 8.9 Smoke collection on `PLANT`

In [12]:
smoke_states = collect_game_states("PLANT")

for s in smoke_states:
    print("=" * 70)
    print("turn:", s.turn)
    print("learner:", s.learner_guess)
    print("feedback:", s.learner_feedback)
    print("expert:", s.expert_guess)
    print("candidates:", s.candidate_count)
    print("repeat:", s.learner_repeated)

turn: 1
learner: CRANE
feedback: BBGGB
expert: RAISE
candidates: 2315
repeat: False
turn: 2
learner: CRANE
feedback: BBGGB
expert: STANK
candidates: 16
repeat: True
turn: 3
learner: CRANE
feedback: BBGGB
expert: STANK
candidates: 16
repeat: True
turn: 4
learner: CRANE
feedback: BBGGB
expert: STANK
candidates: 16
repeat: True
turn: 5
learner: CRANE
feedback: BBGGB
expert: STANK
candidates: 16
repeat: True
turn: 6
learner: CRAZE
feedback: BBGBB
expert: STANK
candidates: 16
repeat: False


## 8.10 Collect 1,000 train answers plus all dev answers

No test answers are used for collection or training.

In [13]:
TRAIN_COLLECTION_LIMIT = 1000

train_answers = [w for w in ANSWERS if answer_split(w) == "train"]
dev_answers = [w for w in ANSWERS if answer_split(w) == "dev"]

rng = random.Random(SEED)
rng.shuffle(train_answers)

train_collection_answers = train_answers[:TRAIN_COLLECTION_LIMIT]
collection_answers = train_collection_answers + dev_answers

print("train collection answers:", len(train_collection_answers))
print("dev collection answers:", len(dev_answers))

train collection answers: 1000
dev collection answers: 231


In [14]:
start = time.perf_counter()
on_policy_states: list[OnPolicyState] = []

for i, answer in enumerate(collection_answers, 1):
    on_policy_states.extend(collect_game_states(answer))

    if i % 100 == 0 or i == len(collection_answers):
        elapsed = time.perf_counter() - start
        print(
            f"{i:4d}/{len(collection_answers)} answers | "
            f"states={len(on_policy_states)} | "
            f"{elapsed:.1f}s"
        )

 100/1231 answers | states=600 | 43.3s
 200/1231 answers | states=1200 | 86.4s
 300/1231 answers | states=1800 | 129.5s
 400/1231 answers | states=2397 | 172.5s
 500/1231 answers | states=2997 | 215.6s
 600/1231 answers | states=3597 | 258.8s
 700/1231 answers | states=4197 | 302.1s
 800/1231 answers | states=4797 | 345.3s
 900/1231 answers | states=5397 | 388.5s
1000/1231 answers | states=5997 | 431.8s
1100/1231 answers | states=6597 | 474.9s
1200/1231 answers | states=7197 | 518.5s
1231/1231 answers | states=7383 | 531.9s


## 8.11 Inspect collection statistics

In [15]:
state_df = pd.DataFrame([asdict(s) for s in on_policy_states])
state_df["relabelable"] = state_df["expert_guess"].notna()
state_df["agrees_with_expert"] = state_df["learner_guess"] == state_df["expert_guess"]

print("states:", len(state_df))
print("relabelable:", int(state_df["relabelable"].sum()))
print("repeat learner actions:", int(state_df["learner_repeated"].sum()))
print("learner/expert agreement:", f"{state_df['agrees_with_expert'].mean():.1%}")
print("\nSPLITS")
print(state_df["split"].value_counts())
print("\nCANDIDATE COUNTS")
print(state_df["candidate_count"].describe(percentiles=[0.5, 0.9, 0.95, 0.99]))

states: 7383
relabelable: 7383
repeat learner actions: 5221
learner/expert agreement: 0.0%

SPLITS
split
train    5997
dev      1386
Name: count, dtype: int64

CANDIDATE COUNTS
count    7383.000000
mean      451.417716
std       836.631146
min         1.000000
50%        65.000000
90%      2315.000000
95%      2315.000000
99%      2315.000000
max      2315.000000
Name: candidate_count, dtype: float64


## 8.12 Build exact deployment prompt → expert action examples

In [16]:
on_policy_examples = []

for state in on_policy_states:
    if state.expert_guess is None:
        continue

    on_policy_examples.append({
        "split": state.split,
        "answer": state.answer,
        "turn": state.turn,
        "candidate_count": state.candidate_count,
        "learner_guess": state.learner_guess,
        "learner_repeated": state.learner_repeated,
        "prompt": state.prompt,
        "response": state.expert_guess,
    })

print("raw on-policy examples:", len(on_policy_examples))

raw on-policy examples: 7383


## 8.13 Deduplicate and reject ambiguity

In [17]:
split_priority = {"train": 0, "dev": 1}
dedup = {}

for ex in on_policy_examples:
    key = (ex["prompt"], ex["response"])
    existing = dedup.get(key)

    if existing is None or split_priority[ex["split"]] > split_priority[existing["split"]]:
        dedup[key] = ex

examples = list(dedup.values())
print("deduplicated on-policy examples:", len(examples))

prompt_targets = {}
for ex in examples:
    prompt_targets.setdefault(ex["prompt"], set()).add(ex["response"])

ambiguous = {p: t for p, t in prompt_targets.items() if len(t) > 1}
print("ambiguous prompts:", len(ambiguous))
assert not ambiguous

examples_df = pd.DataFrame(examples)
prompt_splits = (
    examples_df[["prompt", "split"]]
    .drop_duplicates()
    .groupby("prompt")["split"]
    .nunique()
)
assert prompt_splits.max() == 1
print("No model-facing prompt appears in multiple splits.")
print(examples_df["split"].value_counts())

deduplicated on-policy examples: 600
ambiguous prompts: 0
No model-facing prompt appears in multiple splits.
split
dev      367
train    233
Name: count, dtype: int64


## 8.14 Inspect repetition-correction examples

In [18]:
repeat_examples = examples_df.loc[examples_df["learner_repeated"]]
print("repeated-action training examples:", len(repeat_examples))

sample = repeat_examples.sample(
    min(10, len(repeat_examples)),
    random_state=SEED,
)

for _, row in sample.iterrows():
    print("=" * 80)
    print(row["prompt"])
    print("\nLEARNER:", row["learner_guess"])
    print("EXPERT TARGET:", row["response"])

repeated-action training examples: 515
Play Wordle.

Return exactly one uppercase five-letter English word.
Do not explain.
Do not use punctuation.

Use all previous guesses and feedback when choosing the next guess.
Never repeat a previous guess.

Example valid response:
CRANE

Feedback meanings:
G = correct letter and position
Y = letter is present but wrong position
B = that letter occurrence is not matched

Game history:
C R A N E -> Y Y B B Y
C R A N E -> Y Y B B Y
C R A N E -> Y Y B B Y

LEARNER: CRANE
EXPERT TARGET: RECUR
Play Wordle.

Return exactly one uppercase five-letter English word.
Do not explain.
Do not use punctuation.

Use all previous guesses and feedback when choosing the next guess.
Never repeat a previous guess.

Example valid response:
CRANE

Feedback meanings:
G = correct letter and position
Y = letter is present but wrong position
B = that letter occurrence is not matched

Game history:
C R A N E -> G B B Y Y
C R A N E -> G B B Y Y

LEARNER: CRANE
EXPERT TARGET

## 8.15 Persist on-policy JSONL

In [ ]:
GENERATED_DIR = DATA_DIR / "generated"
GENERATED_DIR.mkdir(parents=True, exist_ok=True)

train_df = examples_df.loc[examples_df["split"] == "train"].copy()
dev_df = examples_df.loc[examples_df["split"] == "dev"].copy()

train_path = GENERATED_DIR / "wordle-onpolicy-train.jsonl"
dev_path = GENERATED_DIR / "wordle-onpolicy-dev.jsonl"

train_df.to_json(train_path, orient="records", lines=True, force_ascii=False)
dev_df.to_json(dev_path, orient="records", lines=True, force_ascii=False)

print("train:", len(train_df), train_path)
print("dev:", len(dev_df), dev_path)

# Phase 2 — Fine-tune Lab 07 on deployment states

## 8.16 Measure sequence lengths

In [ ]:
def render_full_example(prompt: str, response: str) -> tuple[str, str]:
    prompt_text = render_prompt(prompt)
    full_text = prompt_text + response + tokenizer.eos_token
    return prompt_text, full_text

def sequence_length(row) -> int:
    _, full_text = render_full_example(row["prompt"], row["response"])
    return len(tokenizer.encode(full_text, add_special_tokens=False))

max_train_length = max(sequence_length(r) for _, r in train_df.iterrows())
max_dev_length = max(sequence_length(r) for _, r in dev_df.iterrows())

print("max train length:", max_train_length)
print("max dev length:", max_dev_length)

MAX_LENGTH = max(128, max_train_length + 4, max_dev_length + 4)
print("MAX_LENGTH:", MAX_LENGTH)

## 8.17 Response-only encoding and collator

In [ ]:
def encode_example(row) -> dict:
    prompt_text, full_text = render_full_example(row["prompt"], row["response"])
    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
    full_ids = tokenizer(full_text, add_special_tokens=False)["input_ids"]

    if len(full_ids) > MAX_LENGTH:
        raise ValueError(f"{len(full_ids)} > MAX_LENGTH {MAX_LENGTH}")

    labels = [-100] * len(prompt_ids) + full_ids[len(prompt_ids):]

    return {"input_ids": full_ids, "labels": labels}

PAD_ID = tokenizer.pad_token_id or tokenizer.eos_token_id

def collate_batch(rows):
    encoded = [encode_example(row) for row in rows]
    max_len = max(len(x["input_ids"]) for x in encoded)

    input_rows, label_rows, attention_rows = [], [], []

    for item in encoded:
        ids, labels = item["input_ids"], item["labels"]
        pad = max_len - len(ids)

        input_rows.append(ids + [PAD_ID] * pad)
        label_rows.append(labels + [-100] * pad)
        attention_rows.append([1] * len(ids) + [0] * pad)

    return {
        "input_ids": torch.tensor(input_rows, dtype=torch.long),
        "labels": torch.tensor(label_rows, dtype=torch.long),
        "attention_mask": torch.tensor(attention_rows, dtype=torch.long),
    }

BATCH_SIZE = 16
VAL_BATCH_SIZE = 32

train_records = train_df.to_dict("records")
dev_records = dev_df.to_dict("records")

train_loader = DataLoader(
    train_records,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_batch,
    generator=torch.Generator().manual_seed(SEED),
)
val_loader = DataLoader(
    dev_records,
    batch_size=VAL_BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_batch,
)

print("train batches:", len(train_loader))
print("dev batches:", len(val_loader))

## 8.18 Baseline on-policy validation loss

In [ ]:
@torch.no_grad()
def evaluate_loss(model, loader) -> float:
    model.eval()
    losses = []
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        losses.append(outputs.loss.detach().float().cpu().item())
    model.train()
    return float(np.mean(losses))

baseline_onpolicy_val_loss = evaluate_loss(model, val_loader)
print("baseline on-policy validation loss:", baseline_onpolicy_val_loss)

## 8.19 Training configuration

Incremental full-parameter adaptation from Lab 07:

```text
1 epoch
AdamW
learning rate 1e-5
5% warmup
gradient clipping 1.0
```

In [ ]:
LEARNING_RATE = 1e-5
WEIGHT_DECAY = 0.01
EPOCHS = 1

optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

total_steps = len(train_loader) * EPOCHS
warmup_steps = max(1, int(total_steps * 0.05))

def lr_multiplier(step: int) -> float:
    if step < warmup_steps:
        return (step + 1) / warmup_steps
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lr_lambda=lr_multiplier,
)

print("total steps:", total_steps)
print("warmup steps:", warmup_steps)

## 8.20 One-batch sanity check

In [ ]:
batch = next(iter(train_loader))
batch = {k: v.to(device) for k, v in batch.items()}

model.train()
optimizer.zero_grad(set_to_none=True)

outputs = model(**batch)
loss = outputs.loss
loss.backward()

grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

print("sanity loss:", loss.detach().float().cpu().item())
print("gradient norm before clipping:", float(grad_norm))

optimizer.zero_grad(set_to_none=True)

## 8.21 Train on on-policy states

In [ ]:
CHECKPOINT_DIR = Path("../checkpoints/qwen3-0.6b-wordle-onpolicy-sft")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

LOG_EVERY = 25
EVAL_EVERY = max(50, total_steps // 5)

history = []
best_val_loss = float("inf")
global_step = 0
start = time.perf_counter()

model.train()

for epoch in range(EPOCHS):
    for batch in train_loader:
        global_step += 1
        batch = {k: v.to(device) for k, v in batch.items()}

        optimizer.zero_grad(set_to_none=True)
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()

        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        loss_value = loss.detach().float().cpu().item()
        lr = scheduler.get_last_lr()[0]

        history.append({
            "step": global_step,
            "train_loss": loss_value,
            "lr": lr,
            "grad_norm": float(grad_norm),
            "val_loss": None,
        })

        if global_step == 1 or global_step % LOG_EVERY == 0:
            elapsed = time.perf_counter() - start
            print(
                f"step {global_step:4d}/{total_steps} | "
                f"loss {loss_value:.4f} | "
                f"lr {lr:.2e} | "
                f"grad {float(grad_norm):.3f} | "
                f"{elapsed:.1f}s"
            )

        should_eval = global_step % EVAL_EVERY == 0 or global_step == total_steps

        if should_eval:
            val_loss = evaluate_loss(model, val_loader)
            history[-1]["val_loss"] = val_loss
            print(f"  validation @ step {global_step}: {val_loss:.4f}")

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                print("  new best validation loss; saving checkpoint")
                model.save_pretrained(CHECKPOINT_DIR, safe_serialization=True)
                tokenizer.save_pretrained(CHECKPOINT_DIR)

            model.train()

elapsed = time.perf_counter() - start

print("\ntraining complete")
print("elapsed seconds:", elapsed)
print("best validation loss:", best_val_loss)

## 8.22 Plot training

In [ ]:
history_df = pd.DataFrame(history)

plt.figure(figsize=(9, 4))
plt.plot(history_df["step"], history_df["train_loss"], label="train")

val_points = history_df.dropna(subset=["val_loss"])
plt.plot(
    val_points["step"],
    val_points["val_loss"],
    marker="o",
    label="validation",
)

plt.xlabel("Optimizer step")
plt.ylabel("Cross-entropy loss")
plt.title("Lab 08 on-policy SFT")
plt.legend()
plt.show()

## 8.23 Reload the best checkpoint

In [ ]:
del model
if device.type == "mps":
    torch.mps.empty_cache()

model = AutoModelForCausalLM.from_pretrained(
    CHECKPOINT_DIR,
    dtype=torch.float32,
).to(device)
model.eval()

print("reloaded:", CHECKPOINT_DIR)

# Phase 3 — Rerun the frozen Lab 04 benchmark

In Lab 04, change only:

```python
MODEL_ID = "../checkpoints/qwen3-0.6b-wordle-onpolicy-sft"
```

Do not change the prompt, parser, answer set, thinking mode, decoding, or game rules.

First rerun the `PLANT` smoke game, then the full frozen benchmark.

Compare:

```text
Base Qwen
Lab 07 static curriculum SFT
Lab 08 on-policy SFT
```

Record:

- solve rate
- feedback-dependent solves
- valid output rate
- repeat guesses
- history consistency rate
- mean turns on wins

Interpretation:

- repeats ↓, consistency ↑, solve rate ↑ → deployment mismatch was a major cause;
- repeats ↓, consistency ↑, solve rate flat → state use improved, strategy is next;
- training loss improves but gameplay unchanged → one on-policy round is insufficient or the policy target itself needs redesign.

Do not add a second DAgger round until this one is measured.

# Lab 08 checkpoint

Send me, in order:

1. expert relabel for `CRANE -> BBGGB`;
2. `PLANT` collection trace;
3. collection statistics;
4. raw/deduplicated example counts;
5. ambiguous prompt count;
6. train/dev counts;
7. max sequence lengths;
8. baseline on-policy validation loss;
9. sanity loss + gradient norm;
10. training validation checkpoints;
11. post-training `PLANT` trace;
12. full frozen Lab 04 metrics.